In [ ]:
import os
from dotenv import load_dotenv

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


# LangChain

In [ ]:
import os
import sys
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Configuration ---
# Ensure your API key environment variable is set (e.g., OPENAI_API_KEY)
try:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    print(f"Language model initialized: {llm.model_name}")
except Exception as e:
    print(f"Error initializing language model: {e}", file=sys.stderr)
    print("Please ensure your OPENAI_API_KEY is set correctly.", file=sys.stderr)
    sys.exit(1) # Exit if the LLM cannot be initialized


# --- Define Chain Components ---

# 1. Initial Generation: Creates the first draft of the product description.
# The input to this chain will be a dictionary, so we update the prompt template.
generation_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Write a short, simple product description for a new smart coffee mug."),
        ("user", "{product_details}")
    ])
    | llm
    | StrOutputParser()
)

# 2. Critique: Evaluates the generated description and provides feedback.
critique_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """Critique the following product description based on clarity, conciseness, and appeal.
        Provide specific suggestions for improvement."""),
        # This will receive 'initial_description' from the previous step.
        ("user", "Product Description to Critique:\n{initial_description}")
    ])
    | llm
    | StrOutputParser()
)

# 3. Refinement: Rewrites the description based on the original details and the critique.
refinement_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """Based on the original product details and the following critique,
        rewrite the product description to be more effective.

        Original Product Details: {product_details}
        Critique: {critique}

        Refined Product Description:"""),
        ("user", "") # User input is empty as the context is provided in the system message
    ])
    | llm
    | StrOutputParser()
)


# --- Build the Full Reflection Chain (Refactored) ---
# This chain is structured to be more readable and linear.
full_reflection_chain = (
    RunnablePassthrough.assign(
        initial_description=generation_chain
    )
    | RunnablePassthrough.assign(
        critique=critique_chain
    )
    | refinement_chain
)


# --- Run the Chain ---
async def run_reflection_example(product_details: str):
    """Runs the LangChain reflection example with product details."""
    print(f"\n--- Running Reflection Example for Product: '{product_details}' ---")
    try:
        # The chain now expects a dictionary as input from the start.
        final_refined_description = await full_reflection_chain.ainvoke(
            {"product_details": product_details}
        )
        print("\n--- Final Refined Product Description ---")
        print(final_refined_description)
    except Exception as e:
        print(f"\nAn error occurred during chain execution: {e}")

test_product_details = "A mug that keeps coffee hot and can be controlled by a smartphone app."
# The notebook already runs an event loop, so await the coroutine directly.
await run_reflection_example(test_product_details)

# Google ADK

In [ ]:

from google.adk.agents import SequentialAgent, LlmAgent

# The first agent generates the initial draft.
generator = LlmAgent(
    name="DraftWriter",
    description="Generates initial draft content on a given subject.",
    instruction="Write a short, informative paragraph about the user's subject.",
    output_key="draft_text" # The output is saved to this state key.
)

# The second agent critiques the draft from the first agent.
reviewer = LlmAgent(
    name="FactChecker",
    description="Reviews a given text for factual accuracy and provides a structured critique.",
    instruction="""
    You are a meticulous fact-checker.
    1. Read the text provided in the state key 'draft_text'.
    2. Carefully verify the factual accuracy of all claims.
    3. Your final output must be a dictionary containing two keys:
       - "status": A string, either "ACCURATE" or "INACCURATE".
       - "reasoning": A string providing a clear explanation for your status, citing specific issues if any are found.
    """,
    output_key="review_output" # The structured dictionary is saved here.
)

# The SequentialAgent ensures the generator runs before the reviewer.
review_pipeline = SequentialAgent(
    name="WriteAndReview_Pipeline",
    sub_agents=[generator, reviewer]
)

# Execution Flow:
# 1. generator runs -> saves its paragraph to state['draft_text'].
# 2. reviewer runs -> reads state['draft_text'] and saves its dictionary output to state['review_output'].
     

In [ ]:
# Same pipeline, written with the graph-based Workflow API (google-adk 2.x).
# ParallelAgent / SequentialAgent are deprecated in favor of Workflow.
# Requires GOOGLE_API_KEY in .env (loaded above).

from google.adk import Agent, Workflow
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.adk.workflow import JoinNode
from google.genai import types

GEMINI_MODEL = "gemini-flash-latest"

# --- 1. Define Researcher Agents (nodes that run in parallel) ---

researcher_agent_1 = Agent(
    name="RenewableEnergyResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in energy.
Research the latest advancements in 'renewable energy sources'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches renewable energy sources.",
    tools=[google_search],
)

researcher_agent_2 = Agent(
    name="EVResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in transportation.
Research the latest developments in 'electric vehicle technology'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches electric vehicle technology.",
    tools=[google_search],
)

researcher_agent_3 = Agent(
    name="CarbonCaptureResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in climate solutions.
Research the current state of 'carbon capture methods'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches carbon capture methods.",
    tools=[google_search],
)

# --- 2. Join Node (waits for all three researchers, then fans their outputs in) ---
# Its output is a dict keyed by the name of each upstream node.
gather_research = JoinNode(name="GatherResearch")

# --- 3. Function Node (formats the joined outputs for the merger) ---
TOPICS = {
    "RenewableEnergyResearcher": "Renewable Energy",
    "EVResearcher": "Electric Vehicles",
    "CarbonCaptureResearcher": "Carbon Capture",
}


def format_summaries(node_input: dict[str, str]) -> str:
    """Turn the joined researcher outputs into a labeled 'Input Summaries' block."""
    return "\n\n".join(
        f"*   **{topic}:**\n    {node_input[name]}" for name, topic in TOPICS.items()
    )


# --- 4. Define the Merger Agent (the final node of the graph) ---
merger_agent = Agent(
    name="SynthesisAgent",
    model=GEMINI_MODEL,  # Or potentially a more powerful model if needed for synthesis
    instruction="""You are an AI Assistant responsible for combining research findings into a structured report.

Your primary task is to synthesize the research summaries you receive as input, clearly attributing findings to their source areas. Structure your response using headings for each topic. Ensure the report is coherent and integrates the key points smoothly.

**Crucially: Your entire response MUST be grounded *exclusively* on the information provided in the input summaries. Do NOT add any external knowledge, facts, or details not present in these specific summaries.**

**Output Format:**

## Summary of Recent Sustainable Technology Advancements

### Renewable Energy Findings
(Based on RenewableEnergyResearcher's findings)
[Synthesize and elaborate *only* on the renewable energy input summary provided above.]

### Electric Vehicle Findings
(Based on EVResearcher's findings)
[Synthesize and elaborate *only* on the EV input summary provided above.]

### Carbon Capture Findings
(Based on CarbonCaptureResearcher's findings)
[Synthesize and elaborate *only* on the carbon capture input summary provided above.]

### Overall Conclusion
[Provide a brief (1-2 sentence) concluding statement that connects *only* the findings presented above.]

Output *only* the structured report following this format. Do not include introductory or concluding phrases outside this structure, and strictly adhere to using only the provided input summary content.
""",
    description="Combines research findings from the parallel nodes into a structured, cited report, strictly grounded on provided inputs.",
)

# --- 5. Build the Workflow graph ---
# START fans out to the three researchers, which fan back in through the join
# node; the joined result is formatted and handed to the merger agent.
root_agent = Workflow(
    name="ResearchAndSynthesisPipeline",
    edges=[
        (
            "START",
            (researcher_agent_1, researcher_agent_2, researcher_agent_3),
            gather_research,
        ),
        (gather_research, format_summaries, merger_agent),
    ],
)

# --- 6. Run the workflow ---
APP_NAME = "workflow_research_app"
USER_ID = "user_1"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

message = types.Content(
    role="user",
    parts=[types.Part(text="Research the latest sustainable technology advancements.")],
)

async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        text = event.content.parts[0].text
        if text:
            print(f"--- {event.author} ---")
            print(text)
            print()
